# Vault Kubernetes Auth (IAM Role/IRSA) 통합 가이드

이 노트북은 Vault의 AWS 인증 방식을 Kubernetes 환경(EKS)에서 사용하는 방법을 다룹니다. 특히 IRSA(IAM Roles for Service Accounts)를 활용하여 Pod이 Vault에 인증하는 과정을 설명합니다.

1. **AWS Auth 활성화**: Vault에서 AWS 인증 엔진을 마운트합니다.
2. **Role 생성**: 특정 IAM Role ARN을 Vault Role과 매핑합니다.
3. **Vault Secrets Operator (VSO) 연동**: K8s 상의 `VaultAuth`(method: aws)를 통해 IRSA 기반으로 인증하고 시크릿을 동기화합니다.
4. **테스트용 App 배포**: 동기화된 시크릿을 사용하는 예제 앱을 배포하여 확인합니다.

HashiCorp 공식 문서 - https://developer.hashicorp.com/vault/docs/auth/aws

## 환경변수 설정

In [1]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

unset AWS_ACCESS_KEY_ID
unset AWS_SECRET_ACCESS_KEY
unset AWS_SESSION_TOKEN

export SERVICE_NAME="sample-app2"
export NAMESPACE="sample-app2-ns"
export VAULT_NAMESPACE="vault"

# Vault 설정
export VAULT_AUTH_PATH="${SERVICE_NAME}-iam-auth"
export VAULT_ROLE_NAME="${SERVICE_NAME}-role"
export VAULT_POLICY_NAME="${SERVICE_NAME}-policy"

# 연동할 KV 설정
export KV_MOUNT_PATH="my-kv"
export KV_PATH="app/credentials"

# K8s 설정
export K8S_HOST=$(kubectl config view --minify --output 'jsonpath={.clusters[0].cluster.server}')
export SERVICE_ACCOUNT_NAME="${SERVICE_NAME}-sa"

# AWS 설정
export APP_IRSA_ROLE_ARN="arn:aws:iam::341689148868:role/sample-app2-vault-irsa-role"

echo "K8S_HOST: ${K8S_HOST}"
echo "NAMESPACE: ${NAMESPACE}"
echo "VAULT_AUTH_PATH: ${VAULT_AUTH_PATH}"
echo "VAULT_ROLE_NAME: ${VAULT_ROLE_NAME}"
echo "VAULT_POLICY_NAME: ${VAULT_POLICY_NAME}"
echo "SERVICE_ACCOUNT_NAME: ${SERVICE_ACCOUNT_NAME}"
echo "APP_IRSA_ROLE_ARN: ${APP_IRSA_ROLE_ARN}"

direnv: loading ~/workspace/vault-aws/vault-example/.envrc
direnv: export +AWS_ACCESS_KEY_ID +AWS_REGION +AWS_SECRET_ACCESS_KEY +VAULT_ADDR +VAULT_ASSUME_TARGET_ROLE_ARN +VAULT_ROLE_ARN +VAULT_TOKEN
K8S_HOST: https://A450C4D2FB0FA511391CE062E8F78E41.sk1.ap-northeast-2.eks.amazonaws.com
NAMESPACE: sample-app2-ns
VAULT_AUTH_PATH: sample-app2-iam-auth
VAULT_ROLE_NAME: sample-app2-role
VAULT_POLICY_NAME: sample-app2-policy
SERVICE_ACCOUNT_NAME: sample-app2-sa
APP_IRSA_ROLE_ARN: arn:aws:iam::341689148868:role/sample-app2-vault-irsa-role



## 1. 샘플 Application 생성

In [2]:
# Namespace 생성
kubectl create namespace ${NAMESPACE}

# App 이 사용할 ServiceAccount 생성
kubectl apply -f - <<EOF
apiVersion: v1
kind: ServiceAccount
metadata:
  name: ${SERVICE_ACCOUNT_NAME}
  namespace: ${NAMESPACE}
  annotations:
    eks.amazonaws.com/role-arn: ${APP_IRSA_ROLE_ARN}
EOF

# sample-app 생성
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ${SERVICE_NAME}
  namespace: ${NAMESPACE}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: ${SERVICE_NAME}
  template:
    metadata:
      labels:
        app: ${SERVICE_NAME}
    spec:
      serviceAccountName: ${SERVICE_ACCOUNT_NAME}
      containers:
      - name: main
        image: alpine
        command: ["sh", "-c", "while true; do ls -l /etc/secrets; cat /etc/secrets/*; sleep 10; done"]
        volumeMounts:
        - name: secret-volume
          mountPath: /etc/secrets
          readOnly: true
      volumes:
      - name: secret-volume
        secret:
          secretName: ${SERVICE_NAME}-k8s-secret
EOF

Error from server (AlreadyExists): namespaces "sample-app2-ns" already exists
serviceaccount/sample-app2-sa configured
deployment.apps/sample-app2 unchanged



## 2. Vault AWS Auth 설정

In [3]:
# AWS Auth Method 활성화
vault auth enable --path=${VAULT_AUTH_PATH} aws || true

# Vault Role 생성 (IAM 타입)
vault write auth/${VAULT_AUTH_PATH}/role/${VAULT_ROLE_NAME} \
    auth_type=iam \
    bound_iam_principal_arn="${APP_IRSA_ROLE_ARN}" \
    policies="${VAULT_POLICY_NAME}" \
    token_ttl="10h" \
    token_max_ttl="10d" \
    resolve_aws_unique_ids=false

Error enabling aws auth: Error making API request.

Namespace: vault/
URL: POST http://k8s-vault-vaultui-deca82b022-44c49bd5904cf228.elb.ap-northeast-2.amazonaws.com:8200/v1/sys/auth/sample-app2-iam-auth
Code: 400. Errors:

* path is already in use at sample-app2-iam-auth/
Success! Data written to: auth/sample-app2-iam-auth/role/sample-app2-role



In [6]:
# Policy 생성
vault policy write sample-app2-policy - <<EOF
path "${KV_MOUNT_PATH}/data/${KV_PATH}" {
  capabilities = ["read"]
}
EOF

Success! Uploaded policy: sample-app2-policy



## 3. Kubernetes 리소스 생성 (VSO 연동)

VSO가 IRSA를 통해 AWS STS에 인증을 요청하고, 받아온 Identity로 Vault에 로그인합니다.

In [5]:
# Vault Connection 을 관리하는 CRD
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultConnection
metadata:
  name: vault-connection
  namespace: ${NAMESPACE}
spec:
  address: ${VAULT_ADDR}
EOF

# VaultAuth - Vault 인증을 관리하는 CRD
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
  name: ${SERVICE_NAME}-aws-auth
  namespace: ${NAMESPACE}
spec:
  method: aws
  mount: ${VAULT_AUTH_PATH}
  aws:
    role: ${VAULT_ROLE_NAME}
    irsaServiceAccount: ${SERVICE_ACCOUNT_NAME}
  vaultConnectionRef: vault-connection
EOF

# VaultStaticSecret
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  name: ${SERVICE_NAME}-secret
  namespace: ${NAMESPACE}
spec:
  vaultAuthRef: ${SERVICE_NAME}-aws-auth

  mount: ${KV_MOUNT_PATH}
  type: kv-v2
  path: ${KV_PATH}
  destination:
    name: ${SERVICE_NAME}-k8s-secret
    create: true
  refreshAfter: 30s

  rolloutRestartTargets:
  - kind: Deployment
    name: ${SERVICE_NAME}
EOF

vaultconnection.secrets.hashicorp.com/vault-connection created
vaultauth.secrets.hashicorp.com/sample-app2-aws-auth unchanged
vaultstaticsecret.secrets.hashicorp.com/sample-app2-secret unchanged



## 4. 리소스 삭제 (Cleanup)

테스트를 위해 생성한 모든 리소스를 삭제합니다.

In [7]:
# 1. Kubernetes 리소스 삭제
kubectl delete deployment ${SERVICE_NAME} -n ${NAMESPACE}
kubectl delete vaultstaticsecret ${SERVICE_NAME}-secret -n ${NAMESPACE}
kubectl delete vaultauth ${SERVICE_NAME}-aws-auth -n ${NAMESPACE}
kubectl delete vaultconnection vault-connection -n ${NAMESPACE}
kubectl delete serviceaccount ${SERVICE_ACCOUNT_NAME} -n ${NAMESPACE}
kubectl delete namespace ${NAMESPACE}

# 2. Vault Auth 설정 삭제
vault auth disable ${VAULT_AUTH_PATH}

deployment.apps "sample-app2" deleted from sample-app2-ns namespace
vaultstaticsecret.secrets.hashicorp.com "sample-app2-secret" deleted from sample-app2-ns namespace
vaultauth.secrets.hashicorp.com "sample-app2-aws-auth" deleted from sample-app2-ns namespace
vaultconnection.secrets.hashicorp.com "vault-connection" deleted from sample-app2-ns namespace
serviceaccount "sample-app2-sa" deleted from sample-app2-ns namespace
namespace "sample-app2-ns" deleted
Success! Disabled the auth method (if it existed) at: sample-app2-iam-auth/

